# Importation of useful libraries

In [1]:
from PIL import Image, ImageEnhance
from shutil import copytree
import random
import os

# Definitions of augmentation techniques

In order to implement the amount of data we will train our models on, we have decided to implement multiple augmentation techniques, which will be further explained in each section. These will leed to having more differenciated data, and hopefully the model will manage to recognise animals also with lowe quality images thanks to the distortions applied on the training set. You will be able to see that also some randomizations have been applied. This is because we didn't want the dataset to bee too huge or too repetitive. So instead of having any kind of rotation, we will only have one random direction, and instead of having multiple brightness levels, we either make the image brighter or darker, etc.

## Cropped images

The first augmentation technique applied is a random cropping. We had a few concerns about it, because we were afraid that the result of a cropping could lead us to having only grass, sky or water on the pic, so we had to be a bit creative about it. Our first idea was to detect the animals, do a first cropping in order to only have the animal on the picture, and then apply random croppings on the precropped image. Why does this approach not work? The image detection works, in some way, already as a classification. As the models we found are not trained on all the animals we have in our images, they won't be able to detect and crop, for example, an antilope. We were basically putting a step before the classification that needed classification in order to work. After realising this approach would not work, we decided to just keep the size of the images big enough in order to make sure we always have a part of the animal on the cropped image. Taking a quick look at our data, we have seen that the animals are usually pretty large and centered, so taking a size that is half the height and half the width should work well for our purposes. 

In [2]:
def random_half_crop_and_save(image_path, output_folder):
    """Randomly crops an image to half size and saves it to a folder."""
    img = Image.open(image_path)
    width, height = img.size
    new_width, new_height = width // 2, height // 2

    max_x = width - new_width
    max_y = height - new_height
    x, y = random.randint(0, max_x), random.randint(0, max_y)

    cropped_img = img.crop((x, y, x + new_width, y + new_height))

    # Create a unique filename
    original_name, original_ext = os.path.splitext(os.path.basename(image_path))
    new_filename = f"{original_name}_cropped{original_ext}"
    output_path = os.path.join(output_folder, new_filename)
    os.makedirs(output_folder, exist_ok=True)

    cropped_img.save(output_path)



## Stretches

Another interesting augmentation technique is the one creating stretched images. The possible directions are obviously vertical and horizontal, so we created a function that would randomly double either the height or the width of the initial image.

In [3]:
def random_stretch_and_save(image_path, output_folder):
    """Randomly stretches an image and saves it with a unique name."""

    img = Image.open(image_path)
    width, height = img.size

    if random.choice([True, False]):
        new_width = width * 2
        new_height = height
        direction = "horizontal"
    else:
        new_width = width
        new_height = height * 2
        direction = "vertical"

    stretched_img = img.resize((new_width, new_height), Image.LANCZOS)

    # Create a unique filename
    original_name, original_ext = os.path.splitext(os.path.basename(image_path))
    new_filename = f"{original_name}_{direction}_stretched{original_ext}"
    output_path = os.path.join(output_folder, new_filename)
    os.makedirs(output_folder, exist_ok=True)

    stretched_img.save(output_path)


## Rotation

Here we apply a 90° rotation to the image. Again, we want to have some randomness, so the rotation can occur with a 50% chance each either clockwise or counterclockwise.

In [4]:
def random_rotate_and_save(image_path, output_folder):
    """Randomly rotates an image by 90 degrees and saves it with a unique name."""

    img = Image.open(image_path)

    # Decide on rotation direction (clockwise or counterclockwise)
    if random.choice([True, False]):
        rotation_angle = 90
        direction = "clockwise"
    else:
        rotation_angle = -90
        direction = "counterclockwise"

    rotated_img = img.rotate(rotation_angle, expand=True)  # Expand to fit the rotated image

    # Create a unique filename
    original_name, original_ext = os.path.splitext(os.path.basename(image_path))
    new_filename = f"{original_name}_{direction}_rotated{original_ext}"
    output_path = os.path.join(output_folder, new_filename)
    os.makedirs(output_folder, exist_ok=True)

    rotated_img.save(output_path)

## Brightness

The (almost) last technique we apply is a brightness adaptation. Here the image can either become extremely bright or dark. Again this happens with a 50% chance for each.

In [5]:
def random_brightness_adjust_and_save(image_path, output_folder, max_brightness_change=0.9):
    """Randomly adjusts image brightness and saves it with a unique name."""

    img = Image.open(image_path)

    # Randomly choose a brightness factor (between 1-max_brightness_change and 1+max_brightness_change)
    brightness_factor = 1 + random.uniform(-max_brightness_change, max_brightness_change)
    enhancer = ImageEnhance.Brightness(img)
    adjusted_img = enhancer.enhance(brightness_factor)

    # Determine if the image was made brighter or darker
    if brightness_factor > 1:
        change_type = "brighter"
    else:
        change_type = "darker"

    # Create a unique filename
    original_name, original_ext = os.path.splitext(os.path.basename(image_path))
    new_filename = f"{original_name}_{change_type}{original_ext}"
    output_path = os.path.join(output_folder, new_filename)
    os.makedirs(output_folder, exist_ok=True)

    adjusted_img.save(output_path)


## Random combination

The actual last one is a random combination of the 4 techniques above. This will be implemented to each image only once, but there are infinitely many possible outcomes (without the cropping there would be 8 possible different outputs, but given that the possible random croppings are, not infinite because they are limited by the number of pixels, but still a really large number, the number of possible combinaitons is extremely high).

In [6]:
def random_augment_and_save(image_path, output_folder, max_brightness_change=0.9):
    """Applies random brightness, rotation, stretching, and cropping to an image."""

    img = Image.open(image_path)

    # Brightness adjustment
    brightness_factor = 1 + random.uniform(-max_brightness_change, max_brightness_change)
    enhancer = ImageEnhance.Brightness(img)
    img = enhancer.enhance(brightness_factor)

    # Rotation
    rotation_angle = random.choice([90, -90])  
    img = img.rotate(rotation_angle, expand=True)

    # Stretching (either horizontal or vertical)
    if random.choice([True, False]):
        img = img.resize((img.width * 2, img.height), Image.LANCZOS)
    else:
        img = img.resize((img.width, img.height * 2), Image.LANCZOS)

    # Cropping (to half size)
    width, height = img.size
    new_width, new_height = width // 2, height // 2
    max_x = width - new_width
    max_y = height - new_height
    x, y = random.randint(0, max_x), random.randint(0, max_y)
    img = img.crop((x, y, x + new_width, y + new_height))

    # Create a unique filename
    original_name, original_ext = os.path.splitext(os.path.basename(image_path))
    new_filename = f"{original_name}_augmented{original_ext}" 
    output_path = os.path.join(output_folder, new_filename)
    os.makedirs(output_folder, exist_ok=True)

    img.save(output_path)

# Split the data in train and test

We will now split the data in train and test. This is done here for two main reasons: first of all we want the augmentation to be performed only on the train dataset, so it is convenient to do it now; second of all, this gives us the possibility to always have the same training and data test for each model and methodology we use. At first this might seem counterintuitive or counterproductive, but we believe that this could be interesting to have more precise comparisons. Especially since part of our main focus is comparing how well the models perform with augmented training data vs with the normal initial data, we believe that having the same training and testing sets can be extremely significant in our study.

In [11]:
import os
import shutil
from random import shuffle

# Paths to the dataset directory and the train/val directories
dataset_path = "/Users/Elizabeth/Desktop/ML/animals copy"
train_path = "/Users/Elizabeth/Desktop/ML/train"
val_path = "/Users/Elizabeth/Desktop/ML/test"

# Create train and val directories if they don't exist
os.makedirs(train_path, exist_ok=True)
os.makedirs(val_path, exist_ok=True)

# List all class directories in the dataset
class_directories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]

# Iterate over each class directory
for class_dir in class_directories:
    class_path = os.path.join(dataset_path, class_dir)
    images = os.listdir(class_path)
    shuffle(images)  # Shuffle to randomize the files that go into train and val

    # Ensure the destination directories for the class exist
    train_class_path = os.path.join(train_path, class_dir)
    val_class_path = os.path.join(val_path, class_dir)
    os.makedirs(train_class_path, exist_ok=True)
    os.makedirs(val_class_path, exist_ok=True)

    # Move the first 40 images to the train directory and the rest to the val directory
    for i, image in enumerate(images):
        if i < 40:
            shutil.move(os.path.join(class_path, image), os.path.join(train_class_path, image))
        else:
            shutil.move(os.path.join(class_path, image), os.path.join(val_class_path, image))

# Augmentation (only training data)

What happens here precisely is the following. We start by creating the augmented_train folder, which is a copy of the train folder. We do this in order to have all the images of the training data in the augmented folder also in their natural form. Then each of the above explained techniques is applied once to each image. This means that from each image, we will get, in the augmented folder, 6 images: the initial one, a cropped one, a rotaded one, a stretched one, one that is either bright or dark, and one that has a combination of all of these. That means that our training data, that first contained 40 images, now contains 240 ones.

In [20]:
source_folder = "/Users/Elizabeth/Desktop/ML/train"
output_folder = "/Users/Elizabeth/Desktop/ML/augmented_train"
copytree(source_folder, output_folder)

# Apply augmentation to each image in the augmented folder
for folder_name in os.listdir(output_folder):
    if os.path.isdir(os.path.join(output_folder, folder_name)) and not folder_name.startswith('.'):
        folder_path = os.path.join(output_folder, folder_name)
    for image_name in os.listdir(folder_path):
        if image_name.endswith((".jpg", ".jpeg", ".png")):  # Add other extensions if needed
            image_path = os.path.join(folder_path, image_name)
            random_half_crop_and_save(image_path, folder_path)
            random_stretch_and_save(image_path, folder_path)
            random_rotate_and_save(image_path, folder_path)
            random_brightness_adjust_and_save(image_path, folder_path)
            random_augment_and_save(image_path, folder_path)

In order to manage to use this data for training and testing, we need to match each image of an animal with the correct label. For this we create a list of touples, that contains, for each image, the directory of the image and its label. We do this for normal train data, augmented train data and test data.

In [21]:
base_path2= "/Users/Elizabeth/Desktop/ML/animals"
species_folders = os.listdir(base_path2)
print("Folders of species:", species_folders)

Folders of species: ['hippopotamus', 'sparrow', 'gorilla', 'cat', 'rhinoceros', 'wombat', 'seahorse', 'butterfly', 'donkey', 'raccoon', 'dragonfly', 'crab', 'pig', '.DS_Store', 'orangutan', 'turtle', 'antelope', 'dog', 'bee', 'coyote', 'fox', 'pigeon', 'dolphin', 'fly', 'turkey', 'boar', 'goldfish', 'hare', 'bear', 'penguin', 'squid', 'zebra', 'leopard', 'sheep', 'hamster', 'panda', 'mosquito', 'lobster', 'duck', 'ox', 'owl', 'tiger', 'whale', 'crow', 'rat', 'moth', 'eagle', 'reindeer', 'grasshopper', 'otter', 'starfish', 'hyena', 'goat', 'sandpiper', 'seal', 'jellyfish', 'hummingbird', 'mouse', 'hornbill', 'porcupine', 'wolf', 'lizard', 'woodpecker', 'beetle', 'chimpanzee', 'parrot', 'kangaroo', 'pelecaniformes', 'oyster', 'caterpillar', 'okapi', 'ladybugs', 'bat', 'cockroach', 'koala', 'swan', 'octopus', 'hedgehog', 'horse', 'flamingo', 'squirrel', 'bison', 'cow', 'deer', 'lion', 'goose', 'shark', 'snake', 'badger', 'elephant', 'possum']


In [22]:
species_to_label = {species: i for i, species in enumerate(species_folders)}

species_folders = sorted([f for f in os.listdir(base_path2) 
                          if os.path.isdir(os.path.join(base_path2, f)) and not f.startswith('.')])

print("Mapping di specie a label:", species_to_label)

Mapping di specie a label: {'hippopotamus': 0, 'sparrow': 1, 'gorilla': 2, 'cat': 3, 'rhinoceros': 4, 'wombat': 5, 'seahorse': 6, 'butterfly': 7, 'donkey': 8, 'raccoon': 9, 'dragonfly': 10, 'crab': 11, 'pig': 12, '.DS_Store': 13, 'orangutan': 14, 'turtle': 15, 'antelope': 16, 'dog': 17, 'bee': 18, 'coyote': 19, 'fox': 20, 'pigeon': 21, 'dolphin': 22, 'fly': 23, 'turkey': 24, 'boar': 25, 'goldfish': 26, 'hare': 27, 'bear': 28, 'penguin': 29, 'squid': 30, 'zebra': 31, 'leopard': 32, 'sheep': 33, 'hamster': 34, 'panda': 35, 'mosquito': 36, 'lobster': 37, 'duck': 38, 'ox': 39, 'owl': 40, 'tiger': 41, 'whale': 42, 'crow': 43, 'rat': 44, 'moth': 45, 'eagle': 46, 'reindeer': 47, 'grasshopper': 48, 'otter': 49, 'starfish': 50, 'hyena': 51, 'goat': 52, 'sandpiper': 53, 'seal': 54, 'jellyfish': 55, 'hummingbird': 56, 'mouse': 57, 'hornbill': 58, 'porcupine': 59, 'wolf': 60, 'lizard': 61, 'woodpecker': 62, 'beetle': 63, 'chimpanzee': 64, 'parrot': 65, 'kangaroo': 66, 'pelecaniformes': 67, 'oyster

In [23]:
images_with_labels_train = []

for species in species_folders:
    species_path = os.path.join("./train", species)
    label = species_to_label[species]
    for image_file in os.listdir(species_path):
        if image_file.lower().endswith(('.png', '.jpg', '.jpeg')):  # Assicurati che sia un'immagine
            image_path = os.path.join(species_path, image_file)
            images_with_labels_train.append((image_path, label))

In [24]:
images_with_labels_augmented_train = []

for species in species_folders:
    species_path = os.path.join("./augmented_train", species)
    label = species_to_label[species]
    for image_file in os.listdir(species_path):
        if image_file.lower().endswith(('.png', '.jpg', '.jpeg')): 
            image_path = os.path.join(species_path, image_file)
            images_with_labels_train.append((image_path, label))

In [25]:
images_with_labels_augmented_test = []

for species in species_folders:
    species_path = os.path.join("./test", species)
    label = species_to_label[species]
    for image_file in os.listdir(species_path):
        if image_file.lower().endswith(('.png', '.jpg', '.jpeg')): 
            image_path = os.path.join(species_path, image_file)
            images_with_labels_train.append((image_path, label))